In [1]:
import pandas as pd
import numpy as np
from beir import util, LoggingHandler
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
import random
import torch

/work/mbouthil/projects/envs/uwvenv/lib/python3.10/site-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
# Loading Dataset
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

  0%|          | 0/8841823 [00:00<?, ?it/s]

In [3]:
# Limiting the data size
n_queries = 100_000
limited_query_ids = list(queries.keys())[:n_queries]

queries = {qid: queries[qid] for qid in limited_query_ids}

qrels = {qid: qrels[qid] for qid in limited_query_ids if qid in qrels}

# used_doc_ids = set()
# for qid in qrels:
#     used_doc_ids.update(q)

### Passage Exploration

In [4]:
type(corpus)

dict

In [5]:
corpus_texts = [(int(doc_id), corpus[doc_id]["text"]) for doc_id in corpus][:10]

In [6]:
corpus_texts = dict(corpus_texts)

In [7]:
corpus_texts[0]

'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.'

In [8]:
passages = corpus

In [9]:
qid = list(qrels.keys())[0]
print(qid)

1185869


In [10]:
queries[qid]

')what was the immediate impact of the success of the manhattan project?'

In [11]:
pos_pids = [k for k, v in qrels[qid].items() if v > 0]
print(random.choice(pos_pids))

0


### Query Exploration

In [12]:
q_ids = list(queries.keys())
q_ids[:10]

['1185869',
 '1185868',
 '597651',
 '403613',
 '1183785',
 '312651',
 '80385',
 '645590',
 '645337',
 '186154']

In [13]:
queries_text = list(queries.values())[:10]
print(queries_text)

[')what was the immediate impact of the success of the manhattan project?', '_________ justice is designed to repair the harm to victim, the community and the offender caused by the offender criminal act. question 19 options:', 'what color is amber urine', 'is autoimmune hepatitis a bile acid synthesis disorder', 'elegxo meaning', 'how much does an average person make for tutoring', 'can you use a calculator on the compass test', 'what does physical medicine do', 'what does pending mean on listing', 'feeding rice cereal how many times per day']


### Qrels Exploration

In [14]:
keys = list(qrels.keys())[:10]
print(keys)

['1185869', '1185868', '597651', '403613', '1183785', '312651', '80385', '645590', '645337', '186154']


In [15]:
values = list(qrels.values())[:10]
print(values[4])

{'389': 1}


### MS MARCO Dataloader

In [16]:
class MSMARCO:
    def __init__(self,
                 queries:dict,
                 passages:dict, 
                 qrels:dict, 
                 num_negatives:int=8):

        '''Data loader for MS MARCO dataset'''

        self.queries = queries
        self.passages = passages
        self.qrels = qrels
        self.qids = list(self.qrels.keys())
        self.pids = list(self.passages.keys())
        self.num_negatives = num_negatives
    
    def __len__(self):
        return len(self.qrels)
    
    def __getitem__(self, idx):
        qid = self.qids[idx]
        query = self.queries[qid]

        pos_pids = [k for k, v in self.qrels[qid].items() if v > 0]
        pos_passage = self.passages[random.choice(pos_pids)]['text']

        neg_pids = []
        while len(neg_pids) < self.num_negatives:
            pid = random.choice(self.pids)
            if pid not in pos_pids:
                neg_pids.append(pid)

        neg_passages = [self.passages[pid]['text'] for pid in neg_pids]

        return {"query": query, "positive": pos_passage, 'negatives': neg_passages}

In [17]:
dataset = MSMARCO(queries, corpus, qrels)

In [18]:
dataset.__getitem__(0)

{'query': ')what was the immediate impact of the success of the manhattan project?',
 'positive': 'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.',
 'negatives': ['Complex numbers, derivatives, limits, domain-specific vocabulary, writing arguments, and more. Reading skills for grades 3â\x80\x935 are hereâ\x80\x94 plus, phonics for grade 1! Practice now. Dive into middle school science and social studies topics, from genetics to world history and more!',
  'Nearest city with pop. 50,000+: Medford, OR (75.4 miles, pop. 63,154). Nearest city with pop. 200,000+: Northwest Clackamas, OR (245.6 miles, pop. 224,220). Nearest city with pop. 1,000,000+: Los Angeles, CA (638.0 miles, pop. 3,694,820).',
  'May 7, 2015. An 

In [19]:
# Tokenization
def collate_fn(batch, tokenizer, max_length=128):
    queries = [x['query'] for x in batch]
    positives = [x['positive'] for x in batch]
    negatives = []

    for x in batch:
        negatives.extend(x['negatives'])

    q_tok = tokenizer(
        queries, 
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors='pt'
    )

    p_tok = tokenizer(
        positives + negatives,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    return {"query": q_tok, "passages":p_tok}

In [20]:
from torch.utils.data import DataLoader
import os
from dotenv import load_dotenv
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [21]:
dataloader = DataLoader(
    dataset, 
    batch_size=32,
    shuffle=True,
    num_workers=4,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

In [22]:
test = next(iter(dataloader))

In [23]:
test["query"].keys()

KeysView({'input_ids': tensor([[  101,  6210,  1997,  ...,     0,     0,     0],
        [  101,  2054,  2003,  ...,     0,     0,     0],
        [  101,  2779, 26189,  ...,     0,     0,     0],
        ...,
        [  101,  2054,  2024,  ...,     0,     0,     0],
        [  101,  2054,  2024,  ...,     0,     0,     0],
        [  101,  2054,  2221,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})

In [24]:
test["query"]["input_ids"].shape

torch.Size([32, 32])

In [25]:
test["query"]["attention_mask"].shape

torch.Size([32, 32])

In [26]:
test['passages']['input_ids'].shape

torch.Size([288, 128])

# Training

In [27]:
import torch.nn.functional as F
from torch.optim import AdamW
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
import torch.nn as nn
from torch import Tensor

In [28]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class DualEncoder(nn.Module):
    def __init__(self, query_model_name, passage_model_name):
        super().__init__()
        self.query_encoder = AutoModel.from_pretrained(query_model_name)
        self.passage_encoder = AutoModel.from_pretrained(passage_model_name)

    def encode_query(self, **inputs):
        out = self.query_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS
    
    def encode_passage(self, **inputs):
        out = self.passage_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS 

In [29]:
model = DualEncoder(
    query_model_name="bert-base-uncased",
    passage_model_name="bert-base-uncased"
).to(device)

In [ ]:
def contrastive_loss(q_emb:Tensor, p_emb:Tensor, temperature:float=1.0) -> Tensor:

    '''
    Cross Entropy loss give that M_query < M_passage
    '''

    M = q_emb.shape[0]
    N = p_emb.shape[0]

    q_emb = F.normalize(q_emb, dim=-1)
    p_emb = F.normalize(p_emb, dim=-1) # -1 indicates down the last axis. In R^2, that is the columns. 
    
    scores = torch.matmul(q_emb, p_emb.T)/temperature
    labels = torch.arange(M) * int(N/M)
    labels = labels.to(device=scores.device)

    print(scores.shape)
    print(labels)
    print(labels.shape)

    labels.to(scores.device)

    loss = F.cross_entropy(scores, labels)
    return loss

In [37]:
# Testing the proper dimensions
for batch in dataloader:
    q_inputs = {k: v.to(device) for k, v in batch["query"].items()}
    p_inputs = {k: v.to(device) for k, v in batch["passages"].items()}

    q_emb = model.encode_query(**q_inputs)
    p_emb = model.encode_passage(**p_inputs)

    loss = contrastive_loss(q_emb, p_emb)
    break

torch.Size([32, 288])
tensor([  0,   9,  18,  27,  36,  45,  54,  63,  72,  81,  90,  99, 108, 117,
        126, 135, 144, 153, 162, 171, 180, 189, 198, 207, 216, 225, 234, 243,
        252, 261, 270, 279])
torch.Size([32])


In [ ]:
positive_indices.to(scores.device)

In [65]:
# Training Loop
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
epoch = 1
train_loss = []

for i in range(epoch):

    model.train()
    epoch_loss = 0

    for step, batch in enumerate(dataloader):

        q_inputs = {k: v.to(device) for k, v in batch["query"].items()}
        p_inputs = {k: v.to(device) for k, v in batch["passages"].items()}

        q_emb = model.encode_query(**q_inputs)
        p_emb = model.encode_passage(**p_inputs)

        loss = contrastive_loss(q_emb, p_emb)
        epoch_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = epoch_loss/len(dataloader)
    train_loss.append(avg_loss)

: 

In [ ]:
torch.arange(64) * (1 + 1)

tensor([  0,   2,   4,   6,   8,  10,  12,  14,  16,  18,  20,  22,  24,  26,
         28,  30,  32,  34,  36,  38,  40,  42,  44,  46,  48,  50,  52,  54,
         56,  58,  60,  62,  64,  66,  68,  70,  72,  74,  76,  78,  80,  82,
         84,  86,  88,  90,  92,  94,  96,  98, 100, 102, 104, 106, 108, 110,
        112, 114, 116, 118, 120, 122, 124, 126])

In [ ]:
mat3 = torch.tensor([[1, 2, 3], [4, 5, 6]]) # Shape: (2, 3)
vec1 = torch.tensor([1, 2, 3])
mat2 = torch.tensor([[2,3,4], [4,5,6]])

print(mat3)
print(mat2.T)

torch.matmul(mat3, mat2.T)

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[2, 4],
        [3, 5],
        [4, 6]])


tensor([[20, 32],
        [47, 77]])

In [ ]:
mat3 % vec1

tensor([[0, 0, 0],
        [0, 1, 0]])

In [ ]:
for batch in dataloader:
    p_inputs = {k: v.to(device) for k, v in batch["passages"].items()}
    p_embs = model.encode_passage(**p_inputs)
    print(p_embs.shape)
    break

torch.Size([128, 768])


# Full test

In [ ]:
# Training of the Dual Encoders using contrastive loss

import pandas as pd
import numpy as np
from beir.datasets.data_loader import GenericDataLoader
import random
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Loading Dataset
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

# Limiting the data size
n_queries = 100_000
limited_query_ids = list(queries.keys())[:n_queries]

queries = {qid: queries[qid] for qid in limited_query_ids}
qrels = {qid: qrels[qid] for qid in limited_query_ids if qid in qrels}


class MSMARCO:
    def __init__(self,
                 queries:dict,
                 passages:dict, 
                 qrels:dict, 
                 num_negatives:int=8):

        '''Data loader for MS MARCO dataset'''

        self.queries = queries
        self.passages = passages
        self.qrels = qrels
        self.qids = list(self.qrels.keys())
        self.pids = list(self.passages.keys())
        self.num_negatives = num_negatives
    
    def __len__(self):
        return len(self.qrels)
    
    def __getitem__(self, idx):
        qid = self.qids[idx]
        query = self.queries[qid]

        pos_pids = [k for k, v in self.qrels[qid].items() if v > 0]
        pos_passage = self.passages[random.choice(pos_pids)]['text']

        neg_pids = []
        while len(neg_pids) < self.num_negatives:
            pid = random.choice(self.pids)
            if pid not in pos_pids:
                neg_pids.append(pid)

        neg_passages = [self.passages[pid]['text'] for pid in neg_pids]

        return {"query": query, "positive": pos_passage, 'negatives': neg_passages}
    

dataset = MSMARCO(queries, corpus, qrels)


# Tokenization
def collate_fn(batch, tokenizer, max_length=128):
    queries = [x['query'] for x in batch]
    positives = [x['positive'] for x in batch]
    negatives = []

    for x in batch:
        negatives.extend(x['negatives'])

    q_tok = tokenizer(
        queries, 
        padding="max_length",
        truncation=True,
        max_length=32,
        return_tensors='pt'
    )

    p_tok = tokenizer(
        positives + negatives,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    return {"query": q_tok, "passages":p_tok}


tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=64,
    shuffle=True,
    num_workers=4,
    collate_fn=lambda x: collate_fn(x, tokenizer)
)

device = "cuda" if torch.cuda.is_available() else "cpu"

class DualEncoder(nn.Module):
    def __init__(self, query_model_name, passage_model_name):
        super().__init__()
        self.query_encoder = AutoModel.from_pretrained(query_model_name)
        self.passage_encoder = AutoModel.from_pretrained(passage_model_name)

    def encode_query(self, **inputs):
        out = self.query_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS
    
    def encode_passage(self, **inputs):
        out = self.passage_encoder(**inputs)
        return out.last_hidden_state[:, 0]        # CLS 
    

model = DualEncoder(
    query_model_name="bert-base-uncased",
    passage_model_name="bert-base-uncased"
).to(device)


def contrastive_loss(q_emb:Tensor, p_emb:Tensor, temperature:float=1.0) -> Tensor:

    '''
    Cross Entropy loss give that M_query < M_passage
    '''

    M = q_emb.shape[0]
    N = p_emb.shape[1]

    q_emb = F.normalize(q_emb, dim=-1)
    p_emb = F.normalize(p_emb, dim=-1)
    
    scores = torch.matmul(q_emb, p_emb.T)/temperature
    labels = torch.arange(M) * int(128/64)
    labels.to(scores.device)

    loss = F.cross_entropy(scores, labels)
    return loss

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
epoch = 1
train_loss = []

for i in range(epoch):

    model.train()
    epoch_loss = 0

    for step, batch in enumerate(dataloader):

        q_inputs = {k: v.to(device) for k, v in batch["query"].items()}
        p_inputs = {k: v.to(device) for k, v in batch["passages"].items()}

        q_emb = model.encode_query(**q_inputs)
        p_emb = model.encode_passage(**p_inputs)

        loss = contrastive_loss(q_emb, p_emb)
        epoch_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = epoch_loss/len(dataloader)
    train_loss.append(avg_loss)

# plotting Loss
plt.figure(figsize=(12, 12))
plt.suptitle("Bi-Encoder Training Loss")

plt.plot(range(1, len(train_loss)+1), train_loss, label="Training Loss", linestyle="-", marker="o")

plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.style.use('bmh')
plt.savefig("/work/mbouthil/projects/research_project/RAG/figures/loss_curve.png", dpi=300)


# Saving Encoder Weights
save_dir = "/work/mbouthil/projects/research_project/RAG/model_weights"
model.query_encoder.save_pretrained(f"{save_dir}/query_encoder")
model.passage_encoder.save_pretrained(f"{save_dir}/passage_encode3")

tokenizer.save_pretrained(save_dir)

/work/mbouthil/projects/envs/uwvenv/lib/python3.10/site-packages/beir/datasets/data_loader.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/8841823 [00:00<?, ?it/s]